In [21]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


In [22]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="Qwen/Qwen3.5-4B",
    base_url="https://prod-vllm-2.kriyam.ai/v1",
    api_key=os.getenv("VLM_API_TOKEN"),
    temperature=0,
    timeout=120,
    max_retries=2,
    max_tokens=4096,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)

In [23]:




# Output schemas

class PropertyRequirements(BaseModel):
    property_type: str
    bhk: str | None = None
    city: str
    preferred_areas: list[str] = Field(default_factory=list)
    budget_min: int | None = None
    budget_max: int | None = None
    currency: str = "INR"
    mandatory_amenities: list[str] = Field(default_factory=list)
    preferred_amenities: list[str] = Field(default_factory=list)
    connectivity_preferences: list[str] = Field(default_factory=list)


class ResearchTask(BaseModel):
    objective: str
    area: str | None = None
    search_queries: list[str]
    information_to_extract: list[str]


class PlannerOutput(BaseModel):
    requirements: PropertyRequirements
    research_tasks: list[ResearchTask]
    success_criteria: list[str]


parser = PydanticOutputParser(pydantic_object=PlannerOutput)


# Planner prompt

prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a Real Estate Research Planner Agent.

Your responsibility is to understand the customer's request
and prepare a structured research plan for the Research Agent.

Follow these steps:

1. Understand the customer's intent.
2. Extract all property requirements.
3. Separate mandatory requirements from preferences.
4. Identify the preferred locations.
5. Create research tasks for the relevant locations.
6. Generate useful Tavily search queries for each task.
7. Specify exactly what information the Research Agent
   must extract from search results.

Rules:

- Create separate property research tasks for different areas.
- Include budget, BHK and property type in relevant queries.
- Do not invent missing customer requirements.
- Use null for unknown budget values.
- Do not assume an amenity is mandatory unless the customer
  explicitly requires it.
- Include property name, price, location, BHK, amenities,
  source URL and availability in extraction requirements.
- Include connectivity research when requested.
- Avoid duplicate search queries.
- Do not execute searches. Only prepare the research plan.

{format_instructions}
"""),
    ("human", "Customer request:\n{customer_query}")
])


# Planner Agent

def planner_agent(customer_query: str):
    chain = prompt | llm | parser

    result = chain.invoke({
        "customer_query": customer_query,
        "format_instructions": parser.get_format_instructions()
    })

    output = result.model_dump()

    for index, task in enumerate(output["research_tasks"], start=1):
        task["task_id"] = f"T{index}"

    return output


# Test

customer_query = """
Find a 2 BHK apartment in Pune near Hinjewadi or Wakad.
My budget is ₹60-75 lakh.
I need covered parking, a gym and good connectivity to IT parks.
"""

plan = planner_agent(customer_query)

print(plan)


{'requirements': {'property_type': 'Apartment', 'bhk': '2 BHK', 'city': 'Pune', 'preferred_areas': ['Hinjewadi', 'Wakad'], 'budget_min': 6000000, 'budget_max': 7500000, 'currency': 'INR', 'mandatory_amenities': ['Covered Parking', 'Gym'], 'preferred_amenities': [], 'connectivity_preferences': ['Good connectivity to IT parks']}, 'research_tasks': [{'objective': 'Find 2 BHK apartments in Hinjewadi within budget with mandatory amenities', 'area': 'Hinjewadi', 'search_queries': ['2 BHK apartments in Hinjewadi Pune price 60-75 lakh', '2 BHK apartments in Hinjewadi Pune with covered parking and gym', '2 BHK apartments in Hinjewadi Pune near IT parks connectivity'], 'information_to_extract': ['Property name', 'Price', 'Location', 'BHK', 'Amenities', 'Source URL', 'Availability'], 'task_id': 'T1'}, {'objective': 'Find 2 BHK apartments in Wakad within budget with mandatory amenities', 'area': 'Wakad', 'search_queries': ['2 BHK apartments in Wakad Pune price 60-75 lakh', '2 BHK apartments in Wak

In [36]:
from pydantic import BaseModel, Field


class PropertyFinding(BaseModel):
    property_name: str | None = None
    location: str | None = None
    price: str | None = None
    bhk: str | None = None
    area_sqft: str | None = None
    amenities: list[str] = Field(default_factory=list)
    connectivity: list[str] = Field(default_factory=list)
    source_url: list[str] = Field(default_factory=list)
    evidence: list[str] = Field(default_factory=list)
    missing_information: list[str] = Field(default_factory=list)


class ResearchWorkerOutput(BaseModel):
    task_id: str
    task_summary: str
    findings: list[PropertyFinding]
    additional_information_needed: list[str]
    research_complete: bool

from typing import TypedDict, Annotated
import operator


class WorkerState(TypedDict):
    task_id: str
    task: dict
    requirements: dict


class ResearchState(TypedDict):
    planner_output: dict
    worker_results: Annotated[list[dict], operator.add]

In [25]:
from langchain_tavily import TavilySearch
import os

tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

In [26]:
from langgraph.prebuilt import create_react_agent

worker_agent = create_react_agent(
    model=llm,
    tools=[tavily_tool],
    prompt="""
    You are a Real Estate Research Worker.

    You will receive one research task and customer requirements.

    Your responsibilities:

    1. Understand the assigned research objective.
    2. Execute the initial Tavily search queries.
    3. Extract relevant property information.
    4. Compare retrieved information against customer requirements.
    5. Identify missing or unverified information.
    6. Perform additional targeted Tavily searches when necessary.
    7. Avoid repeating previously executed queries.
    8. Return only information supported by search evidence.

    Rules:
    - Never invent property prices or amenities.
    - Do not assume missing information is available.
    - Preserve source URLs.
    - Prioritize relevant results.
    - Stop when sufficient information is collected
      or further searches are unlikely to help.

    Return your research findings clearly.
    """
)

/tmp/ipykernel_1322016/232878546.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  worker_agent = create_react_agent(


In [ ]:
# extractor = llm.with_structured_output(ResearchWorkerOutput)


# def research_worker(state: WorkerState):

#     task = state["task"]

#     prompt = f"""
#     Customer Requirements:
#     {state['requirements']}

#     Assigned Task ID:
#     {state['task_id']}

#     Research Task:
#     {task}

#     Property Fields to Collect:
#     {state['property_fields_to_collect']}

#     Execute this research task using Tavily.
#     You may perform multiple searches if required.

#     Return the collected information, evidence,
#     source URLs and missing information.
#     """

#     response = worker_agent.invoke(
#         {"messages": [{"role": "user", "content": prompt}]},
#         config={"recursion_limit": 12}
#     )

#     research_content = "\n".join(
#         f"{message.type}: {message.content}"
#         for message in response["messages"]
#     )

#     result = extractor.invoke(
#         f"""
#         Convert the following research execution into
#         structured research findings.

#         Assigned Task ID: {state['task_id']}

#         Research Execution:
#         {research_content}

#         Preserve evidence and source URLs.
#         Do not invent missing information.
#         """
#     )

#     output = result.model_dump()
#     output["task_id"] = state["task_id"]

#     return {"worker_results": [output]}

extractor = llm.with_structured_output(ResearchWorkerOutput)

def research_worker(state: WorkerState):

    task = state["task"]

    prompt = f"""
    Customer Requirements:
    {state['requirements']}

    Assigned Task ID:
    {state['task_id']}

    Research Task:
    {task}

    Property Fields to Collect:
    {state['property_fields_to_collect']}

    Execute this research task using Tavily.
    You may perform multiple searches if required.

    Return collected information, evidence,
    source URLs and missing information.
    """

    response = worker_agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 12}
    )

    research_content = "\n".join(
        f"{message.type}: {message.content}"
        for message in response["messages"]
    )

    result = extractor.invoke(f"""
    Convert the research execution into structured findings.

    Task ID: {state['task_id']}

    Research Execution:
    {research_content}

    Preserve evidence and source URLs.
    Do not invent missing information.
    """)

    output = result.model_dump()
    output["task_id"] = state["task_id"]

    return {"worker_results": [output]}

In [28]:
extractor = llm.with_structured_output(ResearchWorkerOutput)


def research_worker(state: WorkerState):

    task = state["task"]

    prompt = f"""
    Customer Requirements:
    {state['requirements']}

    Assigned Task ID:
    {state['task_id']}

    Research Task:
    {task}

    Property Fields to Collect:
    {state['property_fields_to_collect']}

    Execute this research task using Tavily.
    You may perform multiple searches if required.

    Return the collected information, evidence,
    source URLs and missing information.
    """

    response = worker_agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 12}
    )

    research_content = "\n".join(
        f"{message.type}: {message.content}"
        for message in response["messages"]
    )

    result = extractor.invoke(
        f"""
        Convert the following research execution into
        structured research findings.

        Assigned Task ID: {state['task_id']}

        Research Execution:
        {research_content}

        Preserve evidence and source URLs.
        Do not invent missing information.
        """
    )

    output = result.model_dump()
    output["task_id"] = state["task_id"]

    return {"worker_results": [output]}

In [37]:
# from langgraph.types import Send


# def dispatch_research_tasks(state: ResearchState):

#     planner_output = state["planner_output"]

#     requirements = planner_output["requirements"]
#     property_fields = planner_output["property_fields_to_collect"]

#     tasks = planner_output["research_tasks"]

#     return [
#         Send(
#             "research_worker",
#             {
#                 "task_id": task["task_id"],
#                 "task": task,
#                 "requirements": requirements,
#                 "property_fields_to_collect": property_fields
#             }
#         )
#         for task in tasks
#     ]

def dispatch_workers(state: ResearchState):

    plan = state["planner_output"]

    return [
        Send("research_worker", {
            "task_id": task["task_id"],
            "task": task,
            "requirements": plan["requirements"]
        })
        for task in plan["research_tasks"]
    ]

In [38]:
builder = StateGraph(ResearchState)

builder.add_node("research_worker", research_worker)

builder.add_conditional_edges(
    START,
    dispatch_workers,
    ["research_worker"]
)

builder.add_edge("research_worker", END)

research_graph = builder.compile()

In [40]:
research_result = research_graph.invoke({
    "planner_output": plan,
    "worker_results": []
})

KeyError: 'property_fields_to_collect'

In [34]:
plan = planner_agent({"user_query": customer_query, "planner_output": {}})

research_result = research_graph.invoke({
    "planner_output": plan["planner_output"],
    "worker_results": []
})

KeyError: 'planner_output'